In [4]:
import pandas as pd
import numpy as np
import librosa
import soundfile as sf
from pathlib import Path
from tqdm import tqdm
from joblib import Parallel, delayed
import warnings
warnings.filterwarnings('ignore')

COUGHVID_CLEAN = Path("../data/coughvid/coughvid_clean.csv")
CAMBRIDGE_CLEAN = Path("../data/cambridge/cambridge_clean.csv")
OUTPUT_DIR = Path("../data")

In [5]:
cv = pd.read_csv(COUGHVID_CLEAN)
cam = pd.read_csv(CAMBRIDGE_CLEAN)

print(f"COUGHVID rows:   {len(cv)}")
print(f"Cambridge rows:  {len(cam)}")

COUGHVID rows:   16791
Cambridge rows:  68791


In [6]:
def load_audio(path, sr=22050, max_duration=5.0):
    try:
        y, _ = librosa.load(path, sr=sr, mono=True, duration=max_duration)
        # trim leading/trailing silence
        y, _ = librosa.effects.trim(y, top_db=20)
        # skip clips that are too short after trimming
        if len(y) < sr * 0.3:
            return None
        # peak normalise
        if np.abs(y).max() > 0:
            y = y / np.abs(y).max()
        return y
    except Exception:
        return None

In [7]:
def extract_features(path, sr=22050, n_mfcc=13):
    y = load_audio(path, sr=sr)
    if y is None:
        return None

    feats = []

    def agg(arr):
        # arr shape: (n_features, time_frames)
        return np.concatenate([
            arr.mean(axis=1),
            arr.std(axis=1),
            arr.max(axis=1)
        ])

    # --- MFCCs + delta + delta-delta ---
    mfcc   = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    mfcc_d  = librosa.feature.delta(mfcc)
    mfcc_d2 = librosa.feature.delta(mfcc, order=2)
    for arr in [mfcc, mfcc_d, mfcc_d2]:
        feats.append(agg(arr))          # 3 × (13×3) = 117 values

    # --- Spectral features ---
    sc  = librosa.feature.spectral_centroid(y=y, sr=sr)     # (1, T)
    sb  = librosa.feature.spectral_bandwidth(y=y, sr=sr)    # (1, T)
    sro = librosa.feature.spectral_rolloff(y=y, sr=sr)      # (1, T)
    sco = librosa.feature.spectral_contrast(y=y, sr=sr)     # (7, T)
    for arr in [sc, sb, sro]:
        feats.append(agg(arr))          # 3 × (1×3) = 9 values
    feats.append(agg(sco))              # 7×3 = 21 values

    # --- Temporal features ---
    zcr = librosa.feature.zero_crossing_rate(y)             # (1, T)
    rms = librosa.feature.rms(y=y)                          # (1, T)
    for arr in [zcr, rms]:
        feats.append(agg(arr))          # 2 × (1×3) = 6 values

    # --- Chroma ---
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)        # (12, T)
    feats.append(agg(chroma))           # 12×3 = 36 values

    # --- Duration ---
    feats.append(np.array([len(y) / sr]))   # 1 value

    return np.concatenate(feats).astype(np.float32)

In [8]:
# Test on first COUGHVID file
test_path = cv['audio_path'].iloc[0]
print(f"Testing on: {test_path}")

vec = extract_features(test_path)
if vec is not None:
    print(f"Feature vector length: {len(vec)}")
    print(f"First 10 values: {vec[:10].round(3)}")
    print(f"Any NaN: {np.isnan(vec).any()}")
else:
    print("Returned None — check audio path")

Testing on: ..\data\coughvid\audio\coughvid_20211012\00039425-7f3a-42aa-ac13-834aaa2b6b92.webm
Feature vector length: 190
First 10 values: [-167.744  125.43   -30.631   54.838  -44.816   54.547  -34.386    0.524
   -2.393  -20.786]
Any NaN: False


In [9]:
feature_names = []

# MFCCs + delta + delta-delta
for prefix in ['mfcc', 'mfcc_delta', 'mfcc_delta2']:
    for stat in ['mean', 'std', 'max']:
        for i in range(13):
            feature_names.append(f"{prefix}_{i}_{stat}")

# Spectral
for prefix in ['spectral_centroid', 'spectral_bandwidth', 'spectral_rolloff']:
    for stat in ['mean', 'std', 'max']:
        feature_names.append(f"{prefix}_{stat}")

for stat in ['mean', 'std', 'max']:
    for i in range(7):
        feature_names.append(f"spectral_contrast_{i}_{stat}")

# Temporal
for prefix in ['zcr', 'rms']:
    for stat in ['mean', 'std', 'max']:
        feature_names.append(f"{prefix}_{stat}")

# Chroma
for stat in ['mean', 'std', 'max']:
    for i in range(12):
        feature_names.append(f"chroma_{i}_{stat}")

# Duration
feature_names.append('duration')

print(f"Feature names count: {len(feature_names)}")
print(f"First 10: {feature_names[:10]}")
print(f"Last 5:   {feature_names[-5:]}")

Feature names count: 190
First 10: ['mfcc_0_mean', 'mfcc_1_mean', 'mfcc_2_mean', 'mfcc_3_mean', 'mfcc_4_mean', 'mfcc_5_mean', 'mfcc_6_mean', 'mfcc_7_mean', 'mfcc_8_mean', 'mfcc_9_mean']
Last 5:   ['chroma_8_max', 'chroma_9_max', 'chroma_10_max', 'chroma_11_max', 'duration']


In [10]:
def process_row(path, label):
    vec = extract_features(path)
    if vec is None or np.isnan(vec).any():
        return None, None
    return vec, label

def build_feature_matrix(df, path_col, label_col, n_jobs=-1, desc="Extracting"):
    results = Parallel(n_jobs=n_jobs, backend='loky')(
        delayed(process_row)(row[path_col], row[label_col])
        for _, row in tqdm(df.iterrows(), total=len(df), desc=desc)
    )
    
    vectors, labels = zip(*results)
    
    # Filter out None results
    valid = [(v, l) for v, l in zip(vectors, labels) if v is not None]
    print(f"  Successful: {len(valid)} / {len(df)}")
    print(f"  Failed:     {len(df) - len(valid)}")
    
    X = np.vstack([v for v, l in valid])
    y = np.array([l for v, l in valid])
    return X, y

In [11]:
print("Extracting COUGHVID features...")
X_cv, y_cv = build_feature_matrix(cv, 'audio_path', 'label', desc="COUGHVID")

print(f"\nX_cv shape: {X_cv.shape}")
print(f"y_cv shape: {y_cv.shape}")
print(f"COVID-19 %: {y_cv.mean():.1%}")

Extracting COUGHVID features...


COUGHVID: 100%|██████████| 16791/16791 [10:55<00:00, 25.63it/s] 


  Successful: 16213 / 16791
  Failed:     578

X_cv shape: (16213, 190)
y_cv shape: (16213,)
COVID-19 %: 7.9%


In [12]:
print("Extracting Cambridge features...")
X_cam, y_cam = build_feature_matrix(cam, 'audio_path', 'label', desc="Cambridge")

print(f"\nX_cam shape: {X_cam.shape}")
print(f"y_cam shape: {y_cam.shape}")
print(f"COVID-19 %:  {y_cam.mean():.1%}")

Extracting Cambridge features...


Cambridge: 100%|██████████| 68791/68791 [00:38<00:00, 1807.43it/s]


  Successful: 2537 / 68791
  Failed:     66254

X_cam shape: (2537, 190)
y_cam shape: (2537,)
COVID-19 %:  35.8%


In [13]:
# Grab a random Cambridge path and try loading manually
test_path = cam['audio_path'].iloc[5]
print(f"Testing: {test_path}")
print(f"File exists: {Path(test_path).exists()}")

try:
    y, sr = librosa.load(test_path, sr=22050, mono=True, duration=5.0)
    print(f"Loaded OK. Length: {len(y)}, sr: {sr}")
except Exception as e:
    print(f"FAILED: {type(e).__name__}: {e}")

Testing: ..\data\cambridge\audio\covid_data\audio\62-13-e87b6aa7-59d8-4d35-971e62ef7d30a6c4.wav
File exists: True
FAILED: EOFError: 


In [14]:
import random

sample_paths = cam['audio_path'].sample(50, random_state=42).tolist()
failures = 0

for p in sample_paths:
    try:
        y, sr = librosa.load(p, sr=22050, mono=True, duration=1.0)
    except Exception as e:
        failures += 1

print(f"Failed: {failures} / 50")

Failed: 49 / 50


In [15]:
sizes = [Path(p).stat().st_size for p in sample_paths]
print(f"Min size: {min(sizes)} bytes")
print(f"Max size: {max(sizes)} bytes")
print(f"Mean size: {np.mean(sizes):.0f} bytes")
print(f"\nFirst 10 sizes: {sizes[:10]}")

Min size: 0 bytes
Max size: 294956 bytes
Mean size: 5899 bytes

First 10 sizes: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
